# 🤖 Robotics & Automation Tutorial: Kinematics, Path Planning & Control

Welcome to this step-by-step tutorial on **Robotics & Autonomous Automation** in Python using **NumPy**, **SciPy**, and **Matplotlib**.

## 📌 What is Robotics Automation?
Robotics automation combines mechanical modeling, algorithm-driven path planning, sensor perception, and feedback control loops to enable robots to operate autonomously in dynamic environments.

---

## 🛠️ Step 1: Forward Kinematics (2-DOF Planar Robot Arm)

Forward Kinematics calculates the position of the end-effector $(x, y)$ given the joint angles $(\theta_1, \theta_2)$ and arm link lengths $(L_1, L_2)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def forward_kinematics(theta1, theta2, L1=1.0, L2=1.0):
    """Calculates (x, y) position of 2-DOF robotic arm end-effector."""
    # Joint 1 position (base is at 0,0)
    x1 = L1 * np.cos(theta1)
    y1 = L1 * np.sin(theta1)
    
    # Joint 2 (End-effector) position
    x2 = x1 + L2 * np.cos(theta1 + theta2)
    y2 = y1 + L2 * np.sin(theta1 + theta2)
    
    return (0, x1, x2), (0, y1, y2)

# Test joint angles (in radians)
t1, t2 = np.radians(45), np.radians(30)
X, Y = forward_kinematics(t1, t2)

plt.figure(figsize=(6, 6))
plt.plot(X, Y, '-o', linewidth=4, markersize=10, label='Robot Arm Links')
plt.plot(X[-1], Y[-1], 'ro', markersize=12, label=f'End-Effector ({X[-1]:.2f}, {Y[-1]:.2f})')
plt.xlim(-0.5, 2.5)
plt.ylim(-0.5, 2.5)
plt.grid(True)
plt.title("2-DOF Robotic Arm Forward Kinematics")
plt.xlabel("X Position (m)")
plt.ylabel("Y Position (m)")
plt.legend()
plt.show()

--- 
## 🔹 Step 2: Inverse Kinematics (Target Tracking)

Inverse Kinematics determines the required joint angles $(\theta_1, \theta_2)$ to reach a desired target coordinate $(x_{target}, y_{target})$.

In [ ]:
def inverse_kinematics(x_target, y_target, L1=1.0, L2=1.0):
    """Solves inverse kinematics for a 2-DOF planar robot arm."""
    r2 = x_target**2 + y_target**2
    cos_theta2 = (r2 - L1**2 - L2**2) / (2 * L1 * L2)
    
    # Check if target is reachable
    if abs(cos_theta2) > 1.0:
        raise ValueError("Target point is out of reachable workspace!")
        
    theta2 = np.arccos(cos_theta2)
    theta1 = np.arctan2(y_target, x_target) - np.arctan2(L2 * np.sin(theta2), L1 + L2 * np.cos(theta2))
    
    return theta1, theta2

# Set target position
target_x, target_y = 1.2, 0.8
theta1_sol, theta2_sol = inverse_kinematics(target_x, target_y)
X_ik, Y_ik = forward_kinematics(theta1_sol, theta2_sol)

plt.figure(figsize=(6, 6))
plt.plot(X_ik, Y_ik, '-bo', linewidth=4, markersize=8, label='Solved Robot Arm Position')
plt.plot(target_x, target_y, 'g*', markersize=16, label=f'Target ({target_x}, {target_y})')
plt.xlim(-0.5, 2.5)
plt.ylim(-0.5, 2.5)
plt.grid(True)
plt.title("Inverse Kinematics Solution")
plt.legend()
plt.show()

--- 
## 🔹 Step 3: Autonomous Path Planning (A* Search Algorithm)

A* algorithm is widely used in autonomous mobile robots for grid map navigation and shortest-path planning around obstacles.

In [ ]:
import heapq

def astar_search(grid, start, goal):
    """A* path planning on a 2D occupancy grid map."""
    rows, cols = grid.shape
    open_set = []
    heapq.heappush(open_set, (0, start))
    
    came_from = {}
    g_score = {start: 0}
    f_score = {start: np.hypot(start[0]-goal[0], start[1]-goal[1])}
    
    directions = [(0, 1), (1, 0), (0, -1), (-1, 0), (1, 1), (-1, -1), (1, -1), (-1, 1)]
    
    while open_set:
        _, current = heapq.heappop(open_set)
        
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]
            
        for dx, dy in directions:
            neighbor = (current[0] + dx, current[1] + dy)
            if 0 <= neighbor[0] < rows and 0 <= neighbor[1] < cols:
                if grid[neighbor[0], neighbor[1]] == 1:  # Obstacle
                    continue
                
                step_cost = np.hypot(dx, dy)
                tentative_g = g_score[current] + step_cost
                
                if neighbor not in g_score or tentative_g < g_score[neighbor]:
                    came_from[neighbor] = current
                    g_score[neighbor] = tentative_g
                    f_score[neighbor] = tentative_g + np.hypot(neighbor[0]-goal[0], neighbor[1]-goal[1])
                    heapq.heappush(open_set, (f_score[neighbor], neighbor))
                    
    return None

# Create 20x20 Grid Map with obstacles
grid = np.zeros((20, 20))
grid[5:15, 8] = 1   # Vertical wall
grid[12, 3:9] = 1   # Horizontal wall

start_node = (2, 2)
goal_node = (18, 18)
path = astar_search(grid, start_node, goal_node)

# Visualization
plt.figure(figsize=(7, 7))
plt.imshow(grid, cmap='binary', origin='upper')
if path:
    px, py = zip(*path)
    plt.plot(py, px, 'r-o', linewidth=2, markersize=5, label='A* Planned Path')
plt.plot(start_node[1], start_node[0], 'go', markersize=10, label='Robot Start')
plt.plot(goal_node[1], goal_node[0], 'bs', markersize=10, label='Target Goal')
plt.title("Autonomous Robot Path Planning (A* Search)")
plt.legend()
plt.grid(True)
plt.show()

--- 
## 🔹 Step 4: PID Controller for Trajectory Tracking

Proportional-Integral-Derivative (PID) controllers regulate robot actuators to minimize position error during continuous trajectory execution.

In [ ]:
class PIDController:
    def __init__(self, Kp, Ki, Kd):
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.integral = 0
        self.prev_error = 0
        
    def update(self, setpoint, measured_val, dt=0.1):
        error = setpoint - measured_val
        self.integral += error * dt
        derivative = (error - self.prev_error) / dt
        self.prev_error = error
        return self.Kp * error + self.Ki * self.integral + self.Kd * derivative

# Simulate 1D Motor Position Tracking under PID Control
pid = PIDController(Kp=2.5, Ki=0.5, Kd=0.8)
target_pos = 10.0
current_pos = 0.0
dt = 0.05

positions = []
times = np.arange(0, 10, dt)

for t in times:
    control_signal = pid.update(target_pos, current_pos, dt)
    # System dynamics (simplified inertia simulation)
    current_pos += control_signal * dt
    positions.append(current_pos)

plt.figure(figsize=(8, 4))
plt.plot(times, positions, 'b-', label='Robot Position (PID Output)')
plt.axhline(y=target_pos, color='r', linestyle='--', label='Target Position (10m)')
plt.title("Robotic Actuator Trajectory Tracking via PID Controller")
plt.xlabel("Time (seconds)")
plt.ylabel("Position (meters)")
plt.grid(True)
plt.legend()
plt.show()

--- 
## 🎯 Summary

In this tutorial, we implemented:
1. **Forward Kinematics** for calculating robotic arm geometry.
2. **Inverse Kinematics** for targeted end-effector positioning.
3. **A* Path Planning** on occupancy grids for mobile robot navigation.
4. **PID Feedback Control** for precise actuator motion control.

Happy Robotics Automation Coding! 🤖🚀